# 01 — Hyrje, të dhënat dhe validimi

        **Pyetja udhëheqëse:** Si ndërtojmë një eksperiment ML pa mashtruar veten me të dhënat?

        Ky notebook është ndërtuar si një mësim i plotë kompjuterik. Rruga kryesore ndjek:
        **problem konkret → model matematikor → zbatim nga parimet e para → bibliotekë profesionale → krahasim → interpretim**.

        ## Objektivat
        - Të dallohen mostrat, veçoritë dhe ndryshorja objektiv.
- Të ndërtohet ndarja trajnim–validim–provë.
- Të kuptohet standardizimi dhe rrjedhja e informacionit.
- Të përdoret validimi i kryqëzuar në mënyrë korrekte.

        > **Rruga bazë për orën mësimore (45–60 min)** shënohet me seksionet **CORE**.  
        > Seksionet **EXTENDED** dhe **PLAYGROUND** janë për thellim dhe punë të pavarur.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("01 — Hyrje, të dhënat dhe validimi")
print("Seed:", RANDOM_STATE)

# 1. Problem konkret: kalibrimi i një sensori — CORE

Një sensor temperaturash jep një sinjal elektrik që ndryshon me temperaturën reale, por matjet
përmbajnë zhurmë. Çdo rresht i tabelës përfaqëson një **mostër (sample)**; tensioni, derivati lokal
dhe një temperaturë ndihmëse janë **veçori (features)**; temperatura reale është
**ndryshorja objektiv (target variable)**.

Në nxënien e mbikëqyrur (supervised learning) kemi çifte $(\mathbf{x}_i,y_i)$ dhe kërkojmë një
funksion $\hat y=f(\mathbf{x};\theta)$. Qëllimi nuk është të shpjegojmë vetëm të dhënat e njohura,
por të përgjithësojmë te matje të reja.

In [ ]:
n = 320
T_true = rng.uniform(-10, 50, n)
voltage = 0.04*T_true + 1.5 + rng.normal(0, 0.08, n)
aux = 0.3*np.sin(T_true/8) + rng.normal(0, 0.05, n)
X = np.c_[voltage, aux]
y = T_true + rng.normal(0, 0.7, n)

df = pd.DataFrame({"voltage_V": voltage, "aux_signal": aux, "temperature_C": y})
df.head()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df["voltage_V"], df["temperature_C"], s=18)
ax.set_xlabel("Tensioni i sensorit [V]")
ax.set_ylabel("Temperatura reale [°C]")
ax.set_title("Të dhëna eksperimentale me zhurmë")
plt.show()

**Interpretim.** Boshti horizontal është një veçori e matur; boshti vertikal është madhësia që duam të parashikojmë. Përhapja rreth prirjes lineare përfaqëson zhurmë matëse dhe efekte të pamodeluara.

# 2. Ndarja trajnim–validim–provë — CORE

Bashkësia e trajnimit (training set) përdoret për të përshtatur parametrat. Bashkësia e validimit
(validation set) ndihmon në zgjedhjen e modelit ose hiperparametrave. Bashkësia e provës (test set)
duhet të mbetet e paprekur deri në fund.

Kjo ndarje është pjesë e metodës shkencore: modeli testohet mbi të dhëna që nuk kanë marrë pjesë
në vendimet e ndërtimit të tij.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.4, random_state=RANDOM_STATE)
X_valid, X_test, y_valid, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=RANDOM_STATE)

print(X_train.shape, X_valid.shape, X_test.shape)

In [ ]:
fig, ax = plt.subplots()
ax.hist(y_train, bins=18, alpha=0.6, label="trajnim")
ax.hist(y_valid, bins=18, alpha=0.6, label="validim")
ax.hist(y_test, bins=18, alpha=0.6, label="provë")
ax.set_xlabel("Temperatura [°C]")
ax.set_ylabel("Numri i mostrave")
ax.legend()
ax.set_title("Shpërndarja e ndryshores objektiv")
plt.show()

**Interpretim.** Ndarjet nuk duhet të kenë shpërndarje krejt të ndryshme pa arsye fizike. Në probleme reale, ndarja sipas kohës ose sipas instrumentit mund të jetë më e drejtë se një ndarje plotësisht e rastësishme.

# 3. Standardizimi dhe rrjedhja e informacionit — CORE

Standardizimi (standardization) përdor
$$
z_j=\frac{x_j-\mu_j}{\sigma_j}.
$$
Vlerat $\mu_j$ dhe $\sigma_j$ duhet të llogariten **vetëm nga bashkësia e trajnimit**. Nëse
përdorim të gjithë të dhënat përpara ndarjes, informacion nga prova futet në procesin e trajnimit.
Kjo quhet **rrjedhje informacioni (data leakage)**.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

# Gabim: standardizim para ndarjes
scaler_wrong = StandardScaler().fit(X)
X_all_scaled = scaler_wrong.transform(X)
Xtr_w, Xte_w, ytr_w, yte_w = train_test_split(X_all_scaled, y, test_size=0.25, random_state=RANDOM_STATE)
m_wrong = Ridge(alpha=1.0).fit(Xtr_w, ytr_w)
rmse_wrong = mean_squared_error(yte_w, m_wrong.predict(Xte_w))**0.5

# Saktë: scaler përshtatet vetëm mbi trajnim
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)
scaler = StandardScaler().fit(Xtr)
m_right = Ridge(alpha=1.0).fit(scaler.transform(Xtr), ytr)
rmse_right = mean_squared_error(yte, m_right.predict(scaler.transform(Xte)))**0.5
rmse_wrong, rmse_right

In [ ]:
fig, ax = plt.subplots()
ax.bar(["rrjedhje", "pa rrjedhje"], [rmse_wrong, rmse_right])
ax.set_ylabel("RMSE [°C]")
ax.set_title("Një shembull i rrjedhjes së informacionit")
plt.show()

**Interpretim.** Në këtë shembull të thjeshtë diferenca mund të jetë e vogël, por parimi është kritik. Në probleme me përzgjedhje veçorish, imputim ose shumë hiperparametra, rrjedhja mund të japë rezultate dukshëm tepër optimiste.

# 4. Pipeline dhe validim i kryqëzuar — CORE

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

pipe = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = -cross_val_score(pipe, X, y, cv=cv, scoring="neg_root_mean_squared_error")

print("RMSE për çdo fold:", np.round(scores, 3))
print("Mesatarja ± std:", scores.mean(), scores.std())

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.arange(1, len(scores)+1), scores, marker="o")
ax.axhline(scores.mean(), linestyle="--")
ax.set_xlabel("Fold")
ax.set_ylabel("RMSE [°C]")
ax.set_title("Ndryshueshmëria e validimit të kryqëzuar")
plt.show()

**Interpretim.** Një rezultat i vetëm fsheh pasigurinë që vjen nga ndarja e të dhënave. Shpërndarja e rezultateve ndërmjet fold-eve na tregon sa i qëndrueshëm është vlerësimi.

---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 5. Shembull i dytë: të dhëna jo-fizike — EXTENDED

I njëjti disiplinim metodologjik përdoret në mjekësi, financë ose energji: çdo hap parapërpunimi duhet të mësohet vetëm nga të dhënat e trajnimit.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
data = load_breast_cancer()
pipe_cls = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
acc = cross_val_score(pipe_cls, data.data, data.target, cv=5, scoring="accuracy")
acc.mean(), acc.std()

# 6. Diagnostikë: çfarë mund të shkojë keq? — EXTENDED

Gabimet e zakonshme janë: ndarje jo përfaqësuese, rrjedhje informacioni, përdorim i testit për
zgjedhje hiperparametrash dhe ngatërrim i rezultateve të trajnimit me aftësinë e përgjithësimit.

In [ ]:
# Vizualizojmë një ndarje të keqe të qëllimshme sipas vlerës së target-it
order = np.argsort(y)
split = int(0.75*len(y))
y_bad_train = y[order[:split]]
y_bad_test = y[order[split:]]

fig, ax = plt.subplots()
ax.hist(y_bad_train, bins=20, alpha=0.6, label="trajnim i keq")
ax.hist(y_bad_test, bins=20, alpha=0.6, label="provë e keqe")
ax.legend()
ax.set_xlabel("Temperatura [°C]")
ax.set_title("Shembull i domain shift të krijuar nga ndarja")
plt.show()

# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Shpjegoni pse një `Pipeline` parandalon rrjedhjen e informacionit.
2. Ndryshoni raportin trajnim/provë dhe vëzhgoni ndryshimin e RMSE.
3. Krahasoni 5-fold dhe 10-fold cross-validation.
4. Shtoni një veçori plotësisht të rastësishme dhe shihni nëse përmirëson modelin.

        ## B. Ushtrim i plotë
        Ndërtoni një pipeline për një problem klasifikimi, përfshini standardizimin dhe raportoni mesataren dhe devijimin standard të F1 në validim të kryqëzuar.

        ## C. Playground
        Ndryshoni seed, madhësinë e mostrës dhe nivelin e zhurmës së sensorit.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.